# Alias state preparation

In [1]:
from math import log2, ceil
import numpy as np
from guppylang import guppy, comptime
from guppylang.std.quantum import qubit, discard_array, discard
from guppyalgos.algorithms.state_preparation.alias_sampling import alias_samp_prep
from guppylang.std.debug import state_output
from guppyalgos.utils import qarray


Define the probability distribution we wish to load as a numpy array, along with the precision we wish to load, which will determine how many bits are used in the fixed point binary numbers used to create our circuit.

In [2]:
prob_dist = np.array([0.5, 0.25, 0.25])
precision = 0.25 # precision for our loaded probabilities
data_len = len(prob_dist)
n_index_qubits = ceil(log2(data_len))
n_prob_bits = ceil(log2(1/precision))

Generate the alias sampling circuit from the probability distribution and run it

In [5]:
alias_samp_prep_box = alias_samp_prep(prob_dist, precision)

@guppy
def main() -> None:
    index_reg = qarray(n_index_qubits)
    alternative_val_reg = qarray(n_index_qubits)
    compare_alt_reg = qarray(n_prob_bits)
    keep_prob_reg = qarray(n_prob_bits)
    compare_out = qubit()
    alias_samp_prep_box(
        index_reg, alternative_val_reg, keep_prob_reg, compare_alt_reg, compare_out, False
    )
    state_output("result_state", index_reg)
    discard_array(alternative_val_reg)
    discard_array(keep_prob_reg)
    discard_array(compare_alt_reg)
    discard_array(index_reg)
    discard(compare_out)


The probabilities we care about are only on the `index_reg` register, and the state we end up with is

$$\ket{\text{alias}}=\sum_i \sqrt{p_i} \ket{i}\ket{\text{junk}_i}$$

where $\ket{\text{junk}_i}$ is the state of all the other registers other than the first.

We can verify this by taking the partial states over `index_reg`, and checking that the probability distribution we get out is the same as out input.

In [6]:
total_qubits = 3 * n_index_qubits + 2 * n_prob_bits + 5 + int(np.ceil((n_index_qubits - 3) / 2))
res = main.emulator(total_qubits).run()

state = res.partial_state_dicts()[0]["result_state"].state_distribution()
prob_out_circ = np.real(sum(
        [state[i].probability * (state[i].state ** 2) for i in range(len(state))]
))
print(prob_out_circ)


[0.5  0.25 0.25 0.  ]
